# AHAAR — Ingredient-first detector (YOLOv11m)

Trains a **pantry / fridge-scan detector** on ~110 raw ingredient classes:

- 36 produce (Kaggle: `kritikseth/fruit-and-vegetable-image-recognition`)
- 19 Indian spices (Kaggle: `tankitrathaur/indian-spices-images-data`)
- 55 scraped raw ingredients (dals, flours, rice, dairy, oils, leafy, proteins,
  dry fruits, staples) — you scrape these locally with
  `datasets/scripts/scrape_ingredient_images.py`, then upload the resulting
  folder to Google Drive (step 4 below).

Dish-image datasets are deliberately **not** used here — this model's job is
only "what's in the pantry". A separate dish classifier can be trained later.

### Before Run All

1. `Runtime → Change runtime type → T4 GPU`.
2. In the left sidebar 🔑 add Colab secrets (Notebook access **on** for each):
   - `KAGGLE_USERNAME`, `KAGGLE_KEY`
   - `HF_TOKEN` (only needed for the optional step 8)
3. Upload the `scraped/` folder (from your laptop:
   `datasets/raw/ingredient_images/scraped`) to **Google Drive** at
   `MyDrive/AAhar/scraped/` (keep the per-class subfolders, e.g.
   `MyDrive/AAhar/scraped/toor_dal/0001_...jpg`).


## 1. Install

In [ ]:
!pip install -q ultralytics==8.3.* huggingface_hub==0.26.* kaggle six onnx onnxruntime pyyaml 2>&1 | tail -3
import torch
print('CUDA:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory/1e9,1)
      if torch.cuda.is_available() else 'no GPU')

## 2. Config

In [ ]:
REPO_URL    = 'https://github.com/workspace220106/Aahar-Kitchen-System.git'
REPO_BRANCH = 'main'

# Where you placed the scraped/ folder inside Google Drive.
DRIVE_SCRAPED_PATH = '/content/drive/MyDrive/AAhar/scraped'

MODEL     = 'yolo11m.pt'
IMG_SIZE  = 640
EPOCHS    = 60
BATCH     = 16
VAL_SPLIT = 0.10

HF_MODEL_REPO = 'your-username/aahar-ingredient-detector'   # edit for step 8

## 3. Clone repo + set up Kaggle

In [ ]:
import os, json, pathlib, subprocess
from pathlib import Path
from google.colab import userdata

os.chdir('/content')
if not Path('/content/aahar').exists():
    subprocess.run(['git','clone','--branch',REPO_BRANCH,'--depth','1',REPO_URL,
                    '/content/aahar'], check=True)

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
Path('/root/.kaggle').mkdir(exist_ok=True)
Path('/root/.kaggle/kaggle.json').write_text(json.dumps({
    'username': os.environ['KAGGLE_USERNAME'],
    'key':      os.environ['KAGGLE_KEY'],
}))
os.chmod('/root/.kaggle/kaggle.json', 0o600)

manifest = json.load(open('/content/aahar/datasets/processed/class_manifest.json'))
TARGET_CLASSES = set(manifest['ingredient_training_classes'])
print(f'Ingredient training classes: {len(TARGET_CLASSES)}')

## 4. Mount Google Drive → pull scraped ingredients

In [ ]:
import shutil, zipfile
from google.colab import drive
drive.mount('/content/drive')

src_candidates = [
    Path(DRIVE_SCRAPED_PATH),                                # folder
    Path(DRIVE_SCRAPED_PATH + '.zip'),                       # zip next to folder
    Path(DRIVE_SCRAPED_PATH).with_suffix('.zip'),            # /…/scraped.zip
    Path('/content/drive/MyDrive/AAhar/scraped.zip'),        # common default
]
src = next((p for p in src_candidates if p.exists()), None)
assert src is not None, (
    f'Not found. Checked: {[str(p) for p in src_candidates]}. '
    'Upload scraped.zip (preferred) or scraped/ folder to Drive.'
)

dst = Path('/content/raw/scraped')
if dst.exists(): shutil.rmtree(dst)
dst.mkdir(parents=True, exist_ok=True)

if src.suffix.lower() == '.zip':
    print(f'Unzipping {src} ({src.stat().st_size/1e6:.0f} MB)...')
    with zipfile.ZipFile(src) as z: z.extractall(dst.parent)
    # If archive created /content/raw/scraped/scraped/<classes>, flatten it
    nested = dst / 'scraped'
    if nested.is_dir() and not any((dst / p.name).exists() for p in nested.iterdir() if p.is_dir()):
        for p in nested.iterdir(): shutil.move(str(p), dst / p.name)
        nested.rmdir()
else:
    print(f'Copying folder {src}...')
    shutil.copytree(src, dst, dirs_exist_ok=True)

print(f'Scraped classes ready: {len(list(dst.iterdir()))}')
for d in sorted(dst.iterdir())[:10]:
    n = sum(1 for _ in d.iterdir()) if d.is_dir() else 0
    print(f'  {d.name:25s} {n}')
print('  ...')

## 5. Fetch Kaggle ingredient datasets (~3.3 GB)

In [ ]:
# Only the ingredient-image datasets. Dish images are intentionally skipped.
RAW = Path('/content/raw'); RAW.mkdir(exist_ok=True)
DATASETS = [
    ('kritikseth/fruit-and-vegetable-image-recognition', RAW/'fruit-veg'),
    ('tankitrathaur/indian-spices-images-data',          RAW/'indian-spices'),
]
for ref, dst in DATASETS:
    if dst.exists() and any(dst.iterdir()):
        print(f'SKIP (exists): {ref}')
        continue
    dst.mkdir(parents=True, exist_ok=True)
    print(f'--- {ref} ---')
    subprocess.run(['kaggle','datasets','download','-d',ref,'-p',str(dst),'--unzip'], check=True)

subprocess.run(['du','-sh',str(RAW)], check=True)

## 6. Merge + canonicalize → YOLO layout

- Reads `class_manifest.json` to decide which classes are kept.
- Canonicalises folder names across sources (e.g. `carrots` → `carrot`,
  `Asafoetida` → `asafoetida`, `toor_dal` stays as `toor_dal`).
- Generates full-frame YOLO bounding boxes for every classification image.

In [ ]:
import re, random, shutil
import yaml
from PIL import Image

def canon(s: str) -> str:
    s = s.strip().lower().replace('-','').replace('_',' ')
    s = re.sub(r'\s+',' ', s)
    aliases = {
        'carrots':'carrot','raddish':'radish','sweetpotato':'sweet potato',
        'sweetcorn':'sweet corn','jalepeno':'jalapeno','cinnamom stick':'cinnamon stick',
        'rajma':'rajma bean',
    }
    return aliases.get(s, s)

def canon_src_folder_for_scraped(s: str) -> str:
    # scraped folders use underscores -> keep them as-is
    return s.strip().lower().replace('-','_')

# Walk each source and collect (canonical_name -> list of image paths)
from collections import defaultdict
pool = defaultdict(list)

# produce (fruit-veg: train/val/test splits all stacked)
for split in ('train','validation','test'):
    root = Path(f'/content/raw/fruit-veg/{split}')
    if not root.exists(): continue
    for d in root.iterdir():
        if not d.is_dir(): continue
        name = canon(d.name)
        if name not in {canon(x) for x in manifest['produce_ingredients']}: continue
        for p in d.rglob('*'):
            if p.suffix.lower() in {'.jpg','.jpeg','.png','.webp'}:
                pool[name].append(p)

# spices
root = Path('/content/raw/indian-spices/indian-spices-image-data')
if root.exists():
    for d in root.iterdir():
        if not d.is_dir() or d.name.startswith('MetaData'): continue
        name = canon(d.name)
        if name not in {canon(x) for x in manifest['spices']}: continue
        for p in d.rglob('*'):
            if p.suffix.lower() in {'.jpg','.jpeg','.png','.webp'}:
                pool[name].append(p)

# scraped (55)
root = Path('/content/raw/scraped')
if root.exists():
    for d in root.iterdir():
        if not d.is_dir(): continue
        name = canon_src_folder_for_scraped(d.name)   # keep snake_case
        for p in d.iterdir():
            if p.suffix.lower() in {'.jpg','.jpeg','.png','.webp'}:
                pool[name].append(p)

class_names = sorted(pool.keys())
cls_id = {n:i for i,n in enumerate(class_names)}
total = sum(len(v) for v in pool.values())
print(f'Classes kept: {len(class_names)} · Total images: {total}')

# Build YOLO directory layout
DATA = Path('/content/aahar/data')
for sub in ('images/train','images/val','labels/train','labels/val'):
    d = DATA / sub
    if d.exists(): shutil.rmtree(d)
    d.mkdir(parents=True, exist_ok=True)

random.seed(42)
for name, imgs in pool.items():
    random.shuffle(imgs)
    n_val = max(1, int(len(imgs)*VAL_SPLIT))
    for i, img in enumerate(imgs):
        split = 'val' if i < n_val else 'train'
        try:
            with Image.open(img) as im: im.verify()
        except Exception:
            continue
        new = f'{cls_id[name]:03d}_{img.stem[:40]}_{i:05d}{img.suffix.lower()}'
        shutil.copy2(img, DATA / f'images/{split}' / new)
        (DATA / f'labels/{split}' / new.rsplit('.',1)[0]).with_suffix('.txt')\
          .write_text(f'{cls_id[name]} 0.5 0.5 1.0 1.0\n')

cfg_path = Path('/content/aahar/aahar.yaml')
cfg_path.write_text(yaml.safe_dump({
    'path':  str(DATA),
    'train': 'images/train',
    'val':   'images/val',
    'nc':    len(class_names),
    'names': class_names,
}, sort_keys=False))

print('train imgs:', len(list((DATA/'images/train').iterdir())))
print('val   imgs:', len(list((DATA/'images/val').iterdir())))

## 7. Train

In [ ]:
from ultralytics import YOLO
model = YOLO(MODEL)
results = model.train(
    data   = str(cfg_path),
    epochs = EPOCHS,
    imgsz  = IMG_SIZE,
    batch  = BATCH,
    device = 0,
    project= '/content/runs',
    name   = 'aahar_ingredients_v1',
    patience = 15,
    cache  = 'ram',
    amp    = True,
)
best = Path('/content/runs/aahar_ingredients_v1/weights/best.pt')
print('Best:', best, '(exists:', best.exists(), ')')

## 8. Export ONNX + TFLite + push to Hugging Face

In [ ]:
from ultralytics import YOLO
trained = YOLO(str(best))
onnx_path   = trained.export(format='onnx',   imgsz=IMG_SIZE, opset=12, dynamic=False, simplify=True)
tflite_path = trained.export(format='tflite', imgsz=IMG_SIZE, int8=False)
print('ONNX :', onnx_path)
print('TFLite:', tflite_path)

# optional: upload
from huggingface_hub import HfApi, create_repo
token = userdata.get('HF_TOKEN')
if token:
    create_repo(HF_MODEL_REPO, token=token, exist_ok=True, repo_type='model')
    api = HfApi(token=token)
    for p in [best, Path(onnx_path), Path(tflite_path), cfg_path]:
        api.upload_file(path_or_fileobj=str(p), path_in_repo=p.name,
                        repo_id=HF_MODEL_REPO, token=token)
    print(f'Uploaded → https://huggingface.co/{HF_MODEL_REPO}')
else:
    print('HF_TOKEN not set; skipping upload.')